In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import MNIST
from torchvision.transforms import transforms
import torch.optim as optim

In [34]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [35]:
train_set = MNIST(root ="./Assig_data",train = True,transform= transform)
test_set = MNIST(root ="./Assig_data",train = False,transform= transform)

Train_loader = DataLoader(train_set, batch_size = 64, shuffle = True)
Test_loader = DataLoader(test_set, batch_size = 64)

# BUILD THE RNN MODEL

In [36]:
class RNN(nn.Module):
    def __init__(self,input_size ,hidden_size=128, num_layer=1):
        super(RNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layer = num_layer
    # RNN Layer 
        self.rnn = nn.RNN(input_size, hidden_size, num_layer, batch_first = True)
        
    # fully connected layer
        self.fcc = nn.Linear(hidden_size, 10)

    def forward(self, x):
        # optinoal: hidden layer initialization. >> shape(num of layers, batch_size, hidden_size)
        h0 = torch.zeros(self.num_layer, x.size(0), self.hidden_size)

        out,_ = self.rnn(x, h0)
        # 1st value = hidden state of all the timestemps >>(batch, sequence_len, hidden_size)
        # 2st value = hidden state of the last timestemps
        # optinoal: hidden layer initialization. >> shape(num of layers, batch_size, hidden_size)
        out = self.fcc(out[:,-1,:])
        return out
        

        

In [37]:
model = RNN(28)
criterian = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training the model 

In [31]:
epocs = 15
model.train()
for epoc in range(epocs):
    for xb, yb in Train_loader:
        optimizer.zero_grad()
        xb = xb.squeeze(1)
        outputs = model(xb)
        loss = criterian(outputs, yb)
        loss.backward()
        optimizer.step()
    print(f"epoc = {epoc} / {epocs} and loss = {loss.item()}") 

epoc = 0 / 15 and loss = 0.4798707365989685
epoc = 1 / 15 and loss = 0.41065022349357605
epoc = 2 / 15 and loss = 0.3082107603549957
epoc = 3 / 15 and loss = 0.24302345514297485
epoc = 4 / 15 and loss = 0.15938834846019745
epoc = 5 / 15 and loss = 0.22098632156848907
epoc = 6 / 15 and loss = 0.158973827958107
epoc = 7 / 15 and loss = 0.22248820960521698
epoc = 8 / 15 and loss = 0.08913667500019073
epoc = 9 / 15 and loss = 0.05531720444560051
epoc = 10 / 15 and loss = 0.0980326384305954
epoc = 11 / 15 and loss = 0.08579794317483902
epoc = 12 / 15 and loss = 0.11227580904960632
epoc = 13 / 15 and loss = 0.06015937402844429
epoc = 14 / 15 and loss = 0.047063082456588745


In [41]:
model.eval()
with torch.no_grad():
    correct = 0
    tot_val = 0

    for xb, yb in Test_loader:
        xb = xb.squeeze(1)
        outputs = model(xb)
        _, predicted = torch.max(outputs, 1)
        tot_val += yb.size(0)
        correct += (predicted== yb).sum().item()
    print(f"accuracy = {correct/tot_val *100} %")
          

accuracy = 11.58 %
